In [1]:
import json
import os, glob

from bertopic import BERTopic
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

/Users/teamihajlov/Projects/CLIB_TopicVisualization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_jsonl_files(folder_path):
    file_pattern = os.path.join(folder_path, "*.jsonl")
    file_paths = glob.glob(file_pattern)
    
    json_objects = []

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                json_objects.append(json.loads(line))
    
    return json_objects

In [3]:
folder_path = "../cleaned_texts/"
json_data = load_jsonl_files(folder_path)

for obj in json_data[:5]:
    print(obj)

print(len(json_data))

{'text': 'zadruga ideal roman sunce rosa dolina zrak oblak magla raselina povesmo sredina zrak premagrevak šuma zelenilo trava dlan strana planina vrh ključ voda tišina vis bor jela tajac nastupanje vrućina ovca paš hlad lišće šibljika krava mesto koza drveće jezik izdanak usta zalogaj hladovina studenac trava čobanac dečko godina žubor studenac čistina nebo oči nedogled planina glas uzvik glava misao paponja koza moguu čuvarica glas neposlušnost jarac paponja paponjak dečko reč drugarica razmišljanje obrva znak nezadovoljstvo četvrt čas cveće borovina studenac misao dečko nebo učitelj duša zagonetka pamet svetac prilika munja duša glas đavo kozȃ paponja glava krava vreme šuma čobanica jarac noga prutić strana glava jarac paponja šala jahačica pesmica studenac voda torba jela prilika igra samoća reka devojče jarac rog devojka jarac rast stas vrata nebo oči obrva devojče usna poluosmejak oči prava devojče glava strana čuđenje jarac lik tromost posao ruka oči devojče cura prst zemlja vu

In [4]:
docs = []
for data in json_data:
    text = data["text"]
    docs.append(text)

print(docs[0])
print(len(docs))

zadruga ideal roman sunce rosa dolina zrak oblak magla raselina povesmo sredina zrak premagrevak šuma zelenilo trava dlan strana planina vrh ključ voda tišina vis bor jela tajac nastupanje vrućina ovca paš hlad lišće šibljika krava mesto koza drveće jezik izdanak usta zalogaj hladovina studenac trava čobanac dečko godina žubor studenac čistina nebo oči nedogled planina glas uzvik glava misao paponja koza moguu čuvarica glas neposlušnost jarac paponja paponjak dečko reč drugarica razmišljanje obrva znak nezadovoljstvo četvrt čas cveće borovina studenac misao dečko nebo učitelj duša zagonetka pamet svetac prilika munja duša glas đavo kozȃ paponja glava krava vreme šuma čobanica jarac noga prutić strana glava jarac paponja šala jahačica pesmica studenac voda torba jela prilika igra samoća reka devojče jarac rog devojka jarac rast stas vrata nebo oči obrva devojče usna poluosmejak oči prava devojče glava strana čuđenje jarac lik tromost posao ruka oči devojče cura prst zemlja vučenje noga

In [5]:
mode_sbert = SentenceTransformer("distiluse-base-multilingual-cased-v2")

tokenizer_jerteh = AutoTokenizer.from_pretrained("jerteh/Jerteh-355")
model_jerteh = AutoModelForMaskedLM.from_pretrained("jerteh/Jerteh-355")

tokenizer_bert = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model_bert = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
from sentence_transformers import SentenceTransformer

embedding_sbert = SentenceTransformer("distiluse-base-multilingual-cased-v2")
umap_model = UMAP(n_neighbors= 5, n_components= 5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size = 3, metric ='euclidean', cluster_selection_method = 'eom', prediction_data=True)
vectorizer_model = CountVectorizer(min_df = 3, max_df = 0.8, ngram_range = (1, 2))

In [7]:
topic_model = BERTopic(

  language = 'multilingual',
  embedding_model=model_bert,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  min_topic_size = 5,

  top_n_words=20,
  verbose=True
)

topics, probs = topic_model.fit_transform(docs)

topics_bert = topic_model.get_topic_info()

2024-09-05 18:09:53,514 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 4/4 [00:02<00:00,  1.43it/s]
2024-09-05 18:09:58,513 - BERTopic - Embedding - Completed ✓
2024-09-05 18:09:58,514 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2024-09-05 18:10:00,920 - BERTopic - Dimensionality - Completed ✓
2024-09-05 18:10:00,921 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-09-05 18:10:00,932 - BERTopic - Cluster - Completed ✓
2024-09-05 18:10:00,936 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-09-05 18:10:01,844 - BERTopic - Representation - Completed ✓


In [8]:
topics_bert

,Topic,Count,Name,Representation,Representative_Docs
0,-1,32,-1_despot_gđa_kir_nazaren,"[despot, gđa, kir, nazaren, madama, bej, hanum...",[patnica zadruga kola knjiga patnica IGNjATOVI...
1,0,8,0_đeneral_đakon_monah_načelnik,"[đeneral, đakon, monah, načelnik, pukovnik, fr...",[ogrlica PRIČA vreme mesto predgovor kafa gost...
2,1,8,1_arhimandrit_major_patrijarh_čik,"[arhimandrit, major, patrijarh, čik, grof, nas...",[pero novac roman zavod put lekar dan leđa uho...
3,2,8,2_fra_fratar_tatko_naprednjak,"[fra, fratar, tatko, naprednjak, đakon, čovjek...",[SLIКE godina КOMARČIĆ STRANКE venac br КOČIJA...
4,3,6,3_iguman_aga_sag_šator,"[iguman, aga, sag, šator, neje, arhimandrit, v...",[knez grad pripovetka vreme boja kola cena din...
5,4,5,4_frajla_grof_seka_sestrica,"[frajla, grof, seka, sestrica, nami, učiteljic...",[godina avgust mesec godina besnilo veče bura ...
6,5,5,5_zeka_zavrzan_harambaša_sovra,"[zeka, zavrzan, harambaša, sovra, vajat, popa,...",[neverstvo roman izdanje JOVANOVIĆA jutro sto ...
7,6,5,6_iguman_bukvar_gradina_tablica,"[iguman, bukvar, gradina, tablica, tabla, cigl...",[talas kapetan brod zadatak objekt utoka kraj ...
8,7,5,7_sultan_carica_despot_riječ,"[sultan, carica, despot, riječ, savjetnik, zva...",[sultan jelen DIMITRIJEVIĆA sultan hanum hanum...
9,8,5,8_grofica_dragana_grof_urednik,"[grofica, dragana, grof, urednik, drama, gospa...",[zora roman delo LjUBIŠA sad deo odeljak prava...


In [19]:
topics_bert.to_csv('topics_bert_NOUN.csv')

In [10]:
topic_model.visualize_topics()

In [11]:
topic_model.visualize_barchart()

In [14]:
cleaned_docs = topic_model._preprocess_text(docs)

vectorizer = topic_model.vectorizer_model
tokenizer = vectorizer.build_tokenizer()

words = vectorizer.get_feature_names_out()
tokens = [tokenizer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] 
               for topic in range(len(set(topics))-1)]

coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='c_v')
coherence = coherence_model.get_coherence()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [15]:
coherence

0.3394499373165071

In [16]:
top_words_per_topic = topics_bert['Representation'].tolist()

In [17]:
def calculate_topic_diversity(topics, top_n=None):
    if top_n:
        # Trim the topics to the top_n words
        topics = [topic[:top_n] for topic in topics]
    
    unique_words = set()
    total_words = 0
    
    for topic in topics:
        unique_words.update(topic)
        total_words += len(topic)
    
    diversity = len(unique_words) / total_words if total_words > 0 else 0
    return diversity

In [18]:
calculate_topic_diversity(top_words_per_topic, top_n = 10)

0.8466666666666667